# Phase 3 — Build a CNN and train it from scratch

This is the heart of the project. We'll:

1. Define a small CNN **by hand** — a stack of the convolution filters you met in Phase 1.
2. Write the **training loop** yourself: forward → loss → backward → step. Understanding this loop *is* understanding deep learning.
3. Track train vs. validation accuracy each epoch and **watch overfitting happen**.

Don't expect great accuracy here (~70-80%). That's intentional — the gap you'll see between training and validation is the most important lesson in the whole project, and it sets up *why* Phase 4 (transfer learning) works so much better.

In [1]:
import sys; sys.path.insert(0, "../src")  # let us import our own modules
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from data import get_dataloaders
from utils import get_device

torch.manual_seed(42)
device = get_device()
print("Training on:", device)

train_loader, val_loader, test_loader, CLASSES = get_dataloaders(batch_size=32)
print("Classes:", CLASSES)

/Users/mack/miniconda3/envs/nothotdog/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training on: mps


Repo card metadata block was not found. Setting CardData to empty.


Classes: ['hotdog', 'not_hotdog']


## 1. Define the CNN

Our network is two repeats of the same pattern, then a classifier:

```
  Conv -> ReLU -> MaxPool      (learns simple features: edges, colors)
  Conv -> ReLU -> MaxPool      (combines them into bigger features: textures, shapes)
  Flatten -> Linear -> ReLU -> Linear   (decides: hotdog or not)
```

- **Conv2d** is the learnable version of the filter from Phase 1. We start with 3 input channels (RGB) and produce 16, then 32 — each channel is one learned filter.
- **ReLU** keeps positive numbers, zeros out negatives. This non-linearity is what lets the network learn complex things (a stack of pure convolutions would collapse into one).
- **MaxPool** halves the height & width by keeping the strongest value in each 2x2 patch — shrinks the data and grants a little position-invariance.
- **Linear** layers at the end do the actual hotdog/not-hotdog decision, outputting **2 numbers** (one score per class).

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Feature extractor: two conv blocks.
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)   # 3 -> 16 channels
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)  # 16 -> 32 channels
        self.pool = nn.MaxPool2d(2, 2)                            # halves H and W

        # After two pools, a 224x224 image is 56x56 with 32 channels.
        # Classifier head:
        self.fc1 = nn.Linear(32 * 56 * 56, 64)
        self.fc2 = nn.Linear(64, 2)   # 2 outputs: [score_hotdog, score_not_hotdog]

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # block 1
        x = self.pool(F.relu(self.conv2(x)))   # block 2
        x = torch.flatten(x, 1)                # keep batch dim, flatten the rest
        x = F.relu(self.fc1(x))
        x = self.fc2(x)                        # raw scores ("logits"), no softmax needed
        return x


model = SmallCNN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal learnable parameters: {n_params:,}")

### Sanity check: push one batch through

Before training, confirm the shapes line up. Feed one batch in and check we get `[batch, 2]` out. (If the `fc1` size were wrong, this is where you'd find out.)

In [ ]:
images, labels = next(iter(train_loader))
with torch.no_grad():
    out = model(images.to(device))
print("input batch :", tuple(images.shape))
print("output      :", tuple(out.shape), "<- 2 scores per image")
print("example row :", out[0].tolist(), "(raw, untrained -> basically random)")

## 2. Loss and optimizer

- **Loss** measures how wrong the model is. `CrossEntropyLoss` is the standard for classification — it compares the 2 scores against the true label and gives a number we want to make small. (For 2 classes this is exactly binary cross-entropy.)
- **Optimizer** is the rule for nudging the weights to reduce the loss. `Adam` is a reliable default. The **learning rate** (`lr`) is how big each nudge is.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 3. The training loop

This is the engine of deep learning. For each **batch**, five steps repeat:

1. **Forward** — run images through the model to get predictions.
2. **Loss** — measure how wrong they are.
3. **Zero grads** — clear last step's gradients.
4. **Backward** — `loss.backward()` computes, for every weight, which direction reduces the loss (backpropagation).
5. **Step** — `optimizer.step()` nudges every weight a little in that direction.

One pass over all the training data is an **epoch**. We'll also write a no-grad `evaluate` to measure accuracy without learning.

In [ ]:
def train_one_epoch(model, loader):
    model.train()                       # enable training behavior (e.g. dropout, if any)
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)         # 1. forward
        loss = criterion(outputs, labels)  # 2. loss

        optimizer.zero_grad()           # 3. clear old gradients
        loss.backward()                 # 4. backprop: compute gradients
        optimizer.step()                # 5. update the weights


@torch.no_grad()                        # no gradients needed when only measuring
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        total_loss += criterion(outputs, labels).item() * len(labels)
        preds = outputs.argmax(dim=1)   # pick the higher of the 2 scores
        correct += (preds == labels).sum().item()
        total += len(labels)
    return total_loss / total, correct / total

## 4. Train, and record the history

We run several epochs. After each, we measure accuracy on **train** and **validation**. Watch the two numbers: if train accuracy keeps climbing while validation stalls or drops, the model is **overfitting** — memorizing the 160 training images instead of learning what a hotdog is.

In [ ]:
EPOCHS = 15
history = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}

for epoch in range(1, EPOCHS + 1):
    train_one_epoch(model, train_loader)
    tr_loss, tr_acc = evaluate(model, train_loader)
    va_loss, va_acc = evaluate(model, val_loader)

    history["train_acc"].append(tr_acc);  history["val_acc"].append(va_acc)
    history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss)
    print(f"epoch {epoch:2d} | train acc {tr_acc:.2f}  val acc {va_acc:.2f} | "
          f"train loss {tr_loss:.3f}  val loss {va_loss:.3f}")

## 5. Plot the learning curves

A picture makes overfitting obvious. The classic signature: the **train** curve climbs toward 100% while the **validation** curve plateaus well below it, and the gap widens.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history["train_acc"], label="train"); ax[0].plot(history["val_acc"], label="val")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(history["train_loss"], label="train"); ax[1].plot(history["val_loss"], label="val")
ax[1].set_title("Loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.show()

## 6. Save the model

We save the learned weights so we can reload them later without retraining.

In [ ]:
import os
os.makedirs("../models", exist_ok=True)
torch.save(model.state_dict(), "../models/scratch_cnn.pth")
print("saved -> ../models/scratch_cnn.pth")

## What just happened, and what's next

You built a CNN from scratch, wrote the training loop, and trained it. A few things worth sitting with:

- **You almost certainly saw overfitting.** With only 160 images, the network has enough capacity to memorize them. Train accuracy shoots up; validation lags. That gap is the model failing to *generalize*.
- **Things to try** (re-run from the model-creation cell after changing one thing):
  - Add `nn.Dropout(0.5)` before `fc2` — randomly drops connections to fight memorization. Does the gap shrink?
  - Train for more epochs — does validation ever catch up? (Spoiler: not really.)
  - Lower the learning rate to `1e-4`. Smoother curves?

The honest takeaway: a small CNN trained on a few hundred images can only get so far. The fix isn't a cleverer hand-built network — it's **starting from a model that has already seen millions of images.** That's Phase 4: transfer learning, where accuracy jumps to ~95% with *less* training. Onward when you're ready.